# Tasks 1-2 on Kaggle
Settings: Accelerator = GPU, Internet = ON. Private repo: add a Kaggle secret `GH_TOKEN` (read-only, this repo only) and attach it to this notebook. Keep the notebook private.

In [ ]:
import base64, os, subprocess
BRANCH = 'dev'
REPO = 'AfnanRizwan1/GenAI-A1'
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    TOKEN = None

def git(*args):
    """Run git; the token is sent as a one-off header, never stored in .git/config or printed."""
    cmd, secrets = ['git'], []
    if TOKEN:
        b64 = base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()
        cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {b64}']
        secrets = [TOKEN, b64]
    r = subprocess.run(cmd + list(args), capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode:
        raise RuntimeError(f'git {args[0]} failed:\n{out}')
    return out

os.chdir('/kaggle/working')
if not os.path.exists('GenAI-A1'):
    git('clone', '-b', BRANCH, f'https://github.com/{REPO}.git')
os.chdir('GenAI-A1')
git('pull', 'origin', BRANCH)
print(git('log', '--oneline', '-3'))

In [ ]:
!pip install -q pytorch-msssim optuna mlflow onnx onnxruntime opencv-python-headless scikit-learn
!nvidia-smi | head -15
!python -m pytest tests -q 2>&1 | tail -3

In [ ]:
# downloads Oxford-IIIT Pet (~800 MB) and caches 128x128 arrays; runs once
!python -c "from src.data.pets import load_pets; d = load_pets('data'); print({k: v.shape for k, v in d.items()})"

In [ ]:
!TRIALS=8 TRIAL_EPOCHS=5 FINAL_EPOCHS=40 CLS_FINAL_EPOCHS=30 bash scripts/run_t1_t2.sh